# **Project: Age and Emotion Detection through voice**

**Problem Statement:** Description: In this task, you will develop a machine learning model to detect a person’s age from a voice note. The model should only process male voices; if a female voice is detected, it should reject the input and display a message saying, “Upload male voice.” If the person’s age is more than 60, the model should mark them as a senior citizen and detect their emotion. For individuals below 60, the model should only detect their age. Guidelines: This task is designed to test your logic-building and problem-solving skills. We encourage you to embrace the challenge and view it as an opportunity to grow. Please create your own machine learning model and ensure it includes a graphical user interface (GUI). While accuracy is important, we will evaluate your work based on the overall performance of your model and the successful functionality of your GUI.

# Downloading Dataset

In [ ]:
'''
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vedant2022/common-voice-dataset-version-4")

print("Path to dataset files:", path)
'''

In [ ]:
!kaggle datasets download -d vedant2022/common-voice-dataset-version-4 --unzip

# Evaluating Age-Data

In [ ]:
import pandas as pd
import os

# 1. Define the source path
#source_path = '/root/.cache/kagglehub/datasets/vedant2022/common-voice-dataset-version-4/versions/1/train.tsv'
source_path = '/content/data-file/train.tsv'


# Check if file exists
if os.path.exists(source_path):
    # 2. Load the metadata
    df = pd.read_csv(source_path, sep='\t')

    # 3. Clean: Drop rows where age or gender are empty
    # This ensures we have valid data for both attributes
    df = df.dropna(subset=['age', 'gender'])
    df = df[(df['age'] != '') & (df['gender'] != '')]

    # 4. Logic: Calculate required samples
    unique_ages = df['age'].unique()
    total_target = 5000
    samples_per_category = total_target // len(unique_ages)

    print(f"Age categories found: {unique_ages}")
    print(f"Targeting {samples_per_category} samples per category.")

    # 5. Sample: Get balanced samples per age category
    # groupby groups the dataframe by age, and sample picks the subset
    balanced_df = df.groupby('age').apply(
        lambda x: x.sample(n=min(len(x), samples_per_category), random_state=42)
    ).reset_index(drop=True)

    # 6. Save the list of file paths
    balanced_df[['path']].to_csv('files_to_download.csv', index=False)

    print(f"Process complete. 'files_to_download.csv' saved with {len(balanced_df)} files.")
else:
    print(f"Error: File not found at {source_path}")

In [ ]:
import os

# The base directory to search
#base_dir = '/root/.cache/kagglehub/datasets/vedant2022/common-voice-dataset-version-4/versions/1'
base_dir = '/content/data-file/'

# Search recursively
for root, dirs, files in os.walk(base_dir):
    if 'train.tsv' in files:
        print(f"Found it! Path: {os.path.join(root, 'train.tsv')}")
        break
else:
    print("Could not find 'train.tsv' in the specified directory. Please check if the download finished successfully.")

In [ ]:
import pandas as pd

# 1. Load your metadata
#df = pd.read_csv('/root/.cache/kagglehub/datasets/vedant2022/common-voice-dataset-version-4/versions/1/data-file/train.tsv', sep='\t')
df = pd.read_csv('/content/data-file/train.tsv', sep='\t')

# 2. Cleanup: Remove empty values
df = df.dropna(subset=['age', 'gender'])
df = df[(df['age'].str.strip() != '') & (df['gender'].str.strip() != '')]

# 3. Define the custom sampling function
def custom_sample(group):
    # If the group has fewer than 700, take all of them
    if len(group) < 700:
        return group
    # Otherwise, take exactly 700
    else:
        return group.sample(n=700, random_state=42)

# 4. Apply the logic
balanced_df = df.groupby('age', group_keys=False).apply(custom_sample)

# 5. Check the results
print("Final count per age category:")
print(balanced_df['age'].value_counts())

# 6. Save the final list
balanced_df[['path']].to_csv('data_fetch.csv', index=False)
print(f"\nFinal dataset size: {len(balanced_df)} files.")

# Fetching Images from Data

In [ ]:
import os
import shutil
import pandas as pd

# 1. Define your paths
#source_clips_dir = '/root/.cache/kagglehub/datasets/vedant2022/common-voice-dataset-version-4/versions/1/new-clip'
source_clips_dir = '/content/new-clip'
destination_dir = 'master_age_gender'
csv_file = 'data_fetch.csv'

# 2. Create the master folder if it doesn't exist
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)

# 3. Load the file list
df = pd.read_csv(csv_file)
file_list = df['path'].tolist()

# 4. Copy files
print(f"Starting to copy {len(file_list)} files...")

for filename in file_list:
    src = os.path.join(source_clips_dir, filename)
    dst = os.path.join(destination_dir, filename)

    # Check if the file exists in source before copying
    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        print(f"Warning: File not found: {filename}")

print(f"\nFinished! All files are now in the '{destination_dir}' folder.")

# Evaluating Data-Files

In [ ]:
import os
import shutil

# 1. Define source and destination
#source_data_file_dir = '/root/.cache/kagglehub/datasets/vedant2022/common-voice-dataset-version-4/versions/1/data-file'
source_data_file_dir = '/content/data-file'
destination_dir = 'data_files_metadata'

# 2. Create the destination folder
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)

# 3. Copy everything inside 'data-file'
# This will copy train.tsv, test.tsv, dev.tsv, etc.
for item in os.listdir(source_data_file_dir):
    src_path = os.path.join(source_data_file_dir, item)
    dst_path = os.path.join(destination_dir, item)

    if os.path.isfile(src_path):
        shutil.copy2(src_path, dst_path)
        print(f"Copied: {item}")

print(f"\nSuccess! All files from 'data-file' are now in '{destination_dir}'.")

# Creating Matser_data csv

In [ ]:
import pandas as pd

# 1. Define paths
train_tsv_path = '/content/data-file/train.tsv'
fetched_csv_path = 'data_fetch.csv'
output_csv_path = 'master_metadata.csv'

# 2. Load the data
# We load the full train.tsv
master_df = pd.read_csv(train_tsv_path, sep='\t')
# We load the list of files you fetched
fetched_df = pd.read_csv(fetched_csv_path)

# 3. Create the master metadata file
# We filter master_df to keep only rows where the 'path' is in fetched_df['path']
master_metadata = master_df[master_df['path'].isin(fetched_df['path'])]

# 4. Save to CSV
master_metadata.to_csv(output_csv_path, index=False)

print(f"Success! '{output_csv_path}' has been created with {len(master_metadata)} entries.")
print("This file contains all original columns (sentence, age, gender, etc.) for your fetched dataset.")

In [ ]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Load your metadata
metadata_df = pd.read_csv('master_metadata.csv')

# 2. Split the data
# First split: Separate out the Test set (15%)
train_val, test_df = train_test_split(metadata_df, test_size=0.15, stratify=metadata_df['age'], random_state=42)

# Second split: Separate Training (70%) and Validation (15%) from the remainder
# 0.176 * 0.85 = ~0.15 total
train_df, val_df = train_test_split(train_val, test_size=0.176, stratify=train_val['age'], random_state=42)

# 3. Create the directory structure
base_dir = 'voice_age_gender'
splits = {'train': train_df, 'val': val_df, 'test': test_df}

for split_name, df in splits.items():
    split_dir = os.path.join(base_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    # Save the CSV for this split
    df.to_csv(f'{split_name}.csv', index=False)

    # Copy files from master_age_gender to the new split folder
    print(f"Copying {len(df)} files to {split_dir}...")
    for filename in df['path']:
        src = os.path.join('master_age_gender', filename)
        dst = os.path.join(split_dir, filename)
        if os.path.exists(src):
            shutil.copy2(src, dst)
        else:
            print(f"Warning: File {filename} not found in master_age_gender.")

print("\nDataset split complete!")
print(f"Files saved in: {base_dir}")
print("CSVs generated: train.csv, val.csv, test.csv")

In [ ]:
# Downloading Dataset

In [ ]:
import shutil

# 1. Define the folder you want to zip
folder_to_zip = 'voice_age_gender'
# 2. Define the output filename (without the .zip extension)
output_filename = 'voice_age_gender_dataset'

# 3. Create the zip file
# 'zip' is the format, the rest are self-explanatory
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"Successfully zipped '{folder_to_zip}' into '{output_filename}.zip'")

# Custom Model

# Loading Datasets

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# --- 1. Data Preprocessing Pipeline ---
def get_spectrogram(file_path_bytes):
    if hasattr(file_path_bytes, 'numpy'):
        file_path = file_path_bytes.numpy().decode('utf-8')
    else:
        file_path = file_path_bytes.decode('utf-8')

    audio, _ = librosa.load(file_path, sr=22050, duration=3)
    spectrogram = librosa.feature.melspectrogram(y=audio, sr=22050, n_mels=128)
    spectrogram = librosa.power_to_db(spectrogram, ref=np.max)

    # Pad or truncate to (128, 129)
    target_width = 129
    if spectrogram.shape[1] < target_width:
        pad_width = target_width - spectrogram.shape[1]
        spectrogram = np.pad(spectrogram, ((0, 0), (0, pad_width)), mode='constant')
    else:
        spectrogram = spectrogram[:, :target_width]

    return spectrogram[..., np.newaxis].astype(np.float32)

AGE_MAP = {'teens': 0, 'twenties': 1, 'thirties': 2, 'fourties': 3,
           'fifties': 4, 'sixties': 5, 'seventies': 6, 'eighties': 7, 'nineties': 8}

def create_dataset(csv_path, audio_dir):
    df = pd.read_csv(csv_path)
    paths = [os.path.join(audio_dir, str(p)) for p in df['path']]
    genders = [1 if g == 'female' else 0 for g in df['gender']]
    ages = [AGE_MAP[a] for a in df['age']]

    ds = tf.data.Dataset.from_tensor_slices((paths, (genders, ages)))

    def process_path(path, labels):
        gen, age = labels
        spec = tf.py_function(get_spectrogram, [path], Tout=tf.float32)
        spec.set_shape([128, 129, 1])
        return spec, {"gender_output": tf.cast(gen, tf.int64), "age_output": tf.cast(age, tf.int64)}

    return ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

# Load datasets
train_ds = create_dataset('/content/train.csv', '/content/voice_age_gender/train')
val_ds = create_dataset('/content/val.csv', '/content/voice_age_gender/val')

# Verify the shape and consistency
for spec, lbl in train_ds.take(1):
    print(f"Data pipeline output shape: {spec.shape}")

# Building Model & Training

In [ ]:
# --- 2. Build Model ---
inputs = tf.keras.layers.Input(shape=(128, 129, 1))
x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu')(inputs)
x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu')(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)

gender_out = tf.keras.layers.Dense(1, activation='sigmoid', name='gender_output')(x)
age_out = tf.keras.layers.Dense(9, activation='softmax', name='age_output')(x)

model = tf.keras.Model(inputs=inputs, outputs=[gender_out, age_out])
model.compile(
    optimizer='adam',
    loss={'gender_output': 'binary_crossentropy', 'age_output': 'sparse_categorical_crossentropy'},
    metrics={'gender_output': 'accuracy', 'age_output': 'accuracy'}
)

# --- 3. Training & Saving ---

checkpoint = tf.keras.callbacks.ModelCheckpoint('gender_age_model.keras', monitor='val_loss', save_best_only=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[checkpoint])

# Plotting Graphs & Confusion Matrix

In [ ]:
def plot_multi_loss(history):
    # Retrieve loss keys
    gen_loss = history.history['gender_output_loss']
    val_gen_loss = history.history['val_gender_output_loss']
    age_loss = history.history['age_output_loss']
    val_age_loss = history.history['val_age_output_loss']

    plt.figure(figsize=(12, 5))

    # Gender Loss
    plt.subplot(1, 2, 1)
    plt.plot(gen_loss, label='Train Gender Loss')
    plt.plot(val_gen_loss, label='Val Gender Loss')
    plt.title('Gender Output Loss')
    plt.legend(); plt.xlabel('Epochs'); plt.ylabel('Loss')

    # Age Loss
    plt.subplot(1, 2, 2)
    plt.plot(age_loss, label='Train Age Loss')
    plt.plot(val_age_loss, label='Val Age Loss')
    plt.title('Age Output Loss')
    plt.legend(); plt.xlabel('Epochs'); plt.ylabel('Loss')
    plt.show()

plot_multi_loss(history)


def plot_multi_cm(dataset, title, age_labels=list(AGE_MAP.keys())):
    y_true_gen, y_true_age = [], []
    y_pred_gen_list, y_pred_age_list = [], []

    # Collect all true labels and predictions
    for spec, labels in dataset:
        y_true_gen.extend(labels['gender_output'].numpy())
        y_true_age.extend(labels['age_output'].numpy())

        preds = model.predict(spec, verbose=0)
        y_pred_gen_list.extend((preds[0] > 0.5).astype("int32").flatten())
        y_pred_age_list.extend(np.argmax(preds[1], axis=1))

    # Plot Gender CM
    cm_gen = confusion_matrix(y_true_gen, y_pred_gen_list)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_gen, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Male', 'Female'], yticklabels=['Male', 'Female'])
    plt.title(f'{title}: Gender CM')
    plt.show()

    # Plot Age CM
    cm_age = confusion_matrix(y_true_age, y_pred_age_list)
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm_age, annot=True, fmt='d', cmap='Greens',
                xticklabels=age_labels, yticklabels=age_labels)
    plt.title(f'{title}: Age CM')
    plt.show()

# 1. Define the age labels list
# Make sure this matches the order of your AGE_MAP indices
age_label_names = list(AGE_MAP.keys())

print("Evaluating Test Set...")
plot_multi_cm(train_ds, "Train Set", age_labels=age_label_names)

# 2. Call the function for different datasets
print("Evaluating Validation Set...")
plot_multi_cm(val_ds, "Validation Set", age_labels=age_label_names)


# Testing & Visualization

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Load model
model = tf.keras.models.load_model('gender_age_model.keras')

# 2. Load Data
test_ds = create_dataset('/content/test.csv', '/content/voice_age_gender/test')

# 3. Extract Labels and Predictions
y_true_gen, y_true_age = [], []
y_pred_gen, y_pred_age = [], []

print("Generating predictions...")
for spec, labels in test_ds:
    # Extract True labels
    y_true_gen.extend(labels['gender_output'].numpy())
    y_true_age.extend(labels['age_output'].numpy())

    # Predict
    preds = model.predict(spec, verbose=0)
    y_pred_gen.extend((preds[0] > 0.5).astype("int32").flatten())
    y_pred_age.extend(np.argmax(preds[1], axis=1))

# 4. Evaluate Gender Head
print("\n--- Gender Classification Report ---")
print(classification_report(y_true_gen, y_pred_gen, target_names=['Male', 'Female']))

# 5. Evaluate Age Head (Robust to missing classes)
unique_labels = np.unique(np.concatenate([y_true_age, y_pred_age]))
all_age_names = list(AGE_MAP.keys())
present_age_names = [all_age_names[i] for i in unique_labels]

print("\n--- Age Classification Report ---")
print(classification_report(
    y_true_age,
    y_pred_age,
    labels=unique_labels,
    target_names=present_age_names,
    zero_division=0
))

# 6. Confusion Matrix Visualizations
def plot_cm(y_true, y_pred, labels, title, cmap):
    # Ensure cm is sized to the number of present classes
    cm = confusion_matrix(y_true, y_pred, labels=np.unique(np.concatenate([y_true, y_pred])))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

# Visualize
plot_cm(y_true_gen, y_pred_gen, ['Male', 'Female'], 'Gender Confusion Matrix', 'Blues')
plot_cm(y_true_age, y_pred_age, present_age_names, 'Age Confusion Matrix', 'Greens')

# 7. Accuracy Visualization
results = model.evaluate(test_ds, return_dict=True)
plt.figure(figsize=(6, 4))
plt.bar(['Gender Acc', 'Age Acc'], [results['gender_output_accuracy'], results['age_output_accuracy']], color=['blue', 'green'])
plt.ylim(0, 1)
plt.title('Test Set Accuracy per Task')
plt.show()

# Emotion Dataset

# Downloading Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sdeogade/voice-emotion-classification")

print("Path to dataset files:", path)

# Fetching images from Dataset

In [ ]:
import os
import shutil
import random

# 1. Define your paths
source_base_dir = '/root/.cache/kagglehub/datasets/sdeogade/voice-emotion-classification/versions/1/Voice Emotion Dataset/' # UPDATE THIS
destination_dir = 'master_emotion'
emotions = ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad']
files_per_emotion = 850

# 2. Setup
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)

# 3. Process each emotion folder
for emotion in emotions:
    source_folder = os.path.join(source_base_dir, emotion)
    # Create the subfolder inside master_emotion
    target_folder = os.path.join(destination_dir, emotion)
    os.makedirs(target_folder, exist_ok=True)

    if os.path.exists(source_folder):
        all_files = [f for f in os.listdir(source_folder) if f.endswith(('.wav', '.mp3', '.flac'))]
        sample_size = min(len(all_files), files_per_emotion)
        selected_files = random.sample(all_files, sample_size)

        print(f"Processing '{emotion}': Copying {sample_size} files to subfolder...")

        for filename in selected_files:
            src = os.path.join(source_folder, filename)
            dst = os.path.join(target_folder, filename)
            shutil.copy2(src, dst)
    else:
        print(f"Warning: Source folder '{emotion}' not found.")

print(f"\nSuccess! Files copied into labeled subfolders within '{destination_dir}'.")

In [ ]:
import os
import pandas as pd

# 1. Define your path
master_emotion_dir = 'master_emotion'

# 2. Collect data
data = []

# Walk through the directory
for emotion in os.listdir(master_emotion_dir):
    emotion_path = os.path.join(master_emotion_dir, emotion)

    # Ensure it's a directory (the emotion label)
    if os.path.isdir(emotion_path):
        for filename in os.listdir(emotion_path):
            if filename.endswith(('.wav', '.mp3', '.flac')):
                # Create the path relative to the master directory
                file_path = os.path.join(emotion, filename)
                data.append({'path': file_path, 'emotion': emotion})

# 3. Create DataFrame
df = pd.DataFrame(data)

# 4. Save to CSV
df.to_csv('master_emotion.csv', index=False)

print(f"Success! 'master_emotion.csv' created with {len(df)} entries.")
print(df.head())

In [ ]:
import pandas as pd
import os

# 1. Load the existing CSV
df = pd.read_csv('master_emotion.csv')

# 2. Define the new base path

base_path = '/content/master_emotion/'

# 3. Apply the transformation
# We join the base_path with the current relative path
df['path'] = base_path + df['path']

# 4. Save the updated CSV
df.to_csv('master_emotion.csv', index=False)

print("Success! 'master_emotion.csv' has been updated with full paths.")
print(df.head())

# Creating CSVs from Data

In [ ]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

# 1. Load your master metadata
df = pd.read_csv('master_emotion.csv')

# 2. Define the output base directory
base_output_dir = 'voice_emotion'

# 3. Perform splits (using stratify ensures each set has a balanced mix of all emotions)
# First split: 80% train+val, 20% test
train_val, test_df = train_test_split(df, test_size=0.2, stratify=df['emotion'], random_state=42)
# Second split: 80% train, 20% val (from the train_val pool)
train_df, val_df = train_test_split(train_val, test_size=0.25, stratify=train_val['emotion'], random_state=42)

# 4. Save CSV manifests
train_df.to_csv('emotion_train.csv', index=False)
val_df.to_csv('emotion_val.csv', index=False)
test_df.to_csv('emotion_test.csv', index=False)

# 5. Function to copy files into their respective structure
def copy_split_files(dataframe, split_name):
    for _, row in dataframe.iterrows():
        # Source path is already absolute from the previous step
        src = row['path']

        # Construct destination path: voice_emotion/split_name/emotion/filename
        emotion = row['emotion']
        filename = os.path.basename(src)
        dst_folder = os.path.join(base_output_dir, split_name, emotion)
        os.makedirs(dst_folder, exist_ok=True)

        shutil.copy2(src, os.path.join(dst_folder, filename))

# Run the copy process
print("Copying files to training, validation, and testing folders...")
copy_split_files(train_df, 'train')
copy_split_files(val_df, 'val')
copy_split_files(test_df, 'test')

print(f"\nSuccess! Dataset split complete and saved in '{base_output_dir}'.")
print("Manifests generated: emotion_train.csv, emotion_val.csv, emotion_test.csv.")

# Downloading Dataset

In [ ]:
import shutil

# 1. Define the folder you want to zip
folder_to_zip = '/content/voice_emotion'
# 2. Define the output filename (without the .zip extension)
output_filename = 'voice_emotion_dataset'

# 3. Create the zip file
# 'zip' is the format, the rest are self-explanatory
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"Successfully zipped '{folder_to_zip}' into '{output_filename}.zip'")

# Custom Model

# Loading Dataset files

In [ ]:
import tensorflow as tf
import numpy as np
import os
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Configuration
EMOTION_MAP = {'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5}
BASE_DIR = '/content/voice_emotion'

def get_spectrogram(file_path):
    path_str = file_path.numpy().decode('utf-8')
    # librosa.load supports both .wav and .mp3 automatically
    audio, _ = librosa.load(path_str, sr=22050, duration=3)
    spec = librosa.feature.melspectrogram(y=audio, sr=22050, n_mels=128)
    spec = librosa.power_to_db(spec, ref=np.max)
    if spec.shape[1] < 129:
        spec = np.pad(spec, ((0, 0), (0, 129 - spec.shape[1])), mode='constant')
    else:
        spec = spec[:, :129]
    return spec[..., np.newaxis].astype(np.float32)

def create_split_ds(split_name):
    paths, emos, gens = [], [], []
    split_path = os.path.join(BASE_DIR, split_name)

    # Updated: check for multiple extensions
    VALID_EXTENSIONS = ('.wav', '.mp3')

    for emo_name in os.listdir(split_path):
        if emo_name in EMOTION_MAP:
            folder_path = os.path.join(split_path, emo_name)
            for f in os.listdir(folder_path):
                if f.lower().endswith(VALID_EXTENSIONS):
                    paths.append(os.path.join(folder_path, f))
                    emos.append(EMOTION_MAP[emo_name])
                    gens.append(0)

    def process_path(p, g, e):
        spec = tf.py_function(get_spectrogram, [p], tf.float32)
        spec.set_shape([128, 129, 1])
        return spec, {"gender_output": tf.cast(g, tf.int64), "emotion_output": tf.cast(e, tf.int64)}

    ds = tf.data.Dataset.from_tensor_slices((paths, gens, emos))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

# Data Initialization
train_ds = create_split_ds('train')
val_ds = create_split_ds('val')
test_ds = create_split_ds('test')

# Building Model

In [ ]:
# 2. 6-Layer Architecture (3 Blocks of Conv+Pool)
inputs = tf.keras.layers.Input(shape=(128, 129, 1))
x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling2D((2, 2))(x) # Layer 2
x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x) # Layer 3
x = tf.keras.layers.MaxPooling2D((2, 2))(x) # Layer 4
x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x) # Layer 5
x = tf.keras.layers.GlobalAveragePooling2D()(x) # Layer 6

gender_out = tf.keras.layers.Dense(1, activation='sigmoid', name='gender_output')(x)
emo_out = tf.keras.layers.Dense(len(EMOTION_MAP), activation='softmax', name='emotion_output')(x)

model = tf.keras.Model(inputs=inputs, outputs=[gender_out, emo_out])
model.compile(optimizer='adam',
              loss={'gender_output': 'binary_crossentropy', 'emotion_output': 'sparse_categorical_crossentropy'},
              metrics={'emotion_output': 'accuracy'})

# Training & Visualization

In [ ]:
# 3. Training
train_ds = create_split_ds('train')
val_ds = create_split_ds('val')
test_ds = create_split_ds('test')

history = model.fit(train_ds, validation_data=val_ds, epochs=20,
                    callbacks=[tf.keras.callbacks.ModelCheckpoint('emotion_model.keras', save_best_only=True)])

# 4. Diagnostics
plt.plot(history.history['emotion_output_accuracy'], label='Train Acc')
plt.plot(history.history['val_emotion_output_accuracy'], label='Val Acc')
plt.title('Training Performance')
plt.legend()
plt.show()

def plot_cm(ds, title):
    y_true, y_pred = [], []
    for spec, labs in ds:
        y_true.extend(labs['emotion_output'].numpy())
        preds = model.predict(spec, verbose=0)[1]
        y_pred.extend(np.argmax(preds, axis=1))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=EMOTION_MAP.keys(), yticklabels=EMOTION_MAP.keys())
    plt.title(f'CM: {title}')
    plt.show()

plot_cm(train_ds, "Train")
plot_cm(val_ds, "Val")

# Testing & Visualization

In [ ]:
# 1. Load the Best Model
model = tf.keras.models.load_model('emotion_model.keras')

# 2. Use your existing create_split_ds for the test folder
test_ds = create_split_ds('test')

# 1. Calculate Test Loss and Accuracy
# Note: model.evaluate returns [total_loss, gender_loss, emotion_loss, emotion_acc]
test_results = model.evaluate(test_ds, verbose=1)
test_loss = test_results[0]
test_accuracy = test_results[-1] # Usually the last index in a compiled model

print(f"\n--- Final Test Metrics ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# 2. Visualize Test Performance vs History
plt.figure(figsize=(10, 5))

# Plot Test Accuracy as a single point on the History plot
plt.plot(history.history['emotion_output_accuracy'], label='Train Accuracy')
plt.plot(history.history['val_emotion_output_accuracy'], label='Val Accuracy')
plt.axhline(y=test_accuracy, color='r', linestyle='--', label='Test Accuracy')
plt.title('Accuracy: Train vs Val vs Test')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# 3. Comprehensive Evaluation
def evaluate_and_plot_cm(ds, model, title="Test Set"):
    y_true, y_pred = [], []

    # We iterate through the whole test dataset
    for spec, labs in ds:
        # Get predictions from the emotion head (index 1)
        preds = model.predict(spec, verbose=0)[1]
        y_true.extend(labs['emotion_output'].numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    # Classification Report
    print(f"\n--- {title} Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=EMOTION_MAP.keys()))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=EMOTION_MAP.keys(),
                yticklabels=EMOTION_MAP.keys(),
                cmap='Blues')
    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

# Run the evaluation
evaluate_and_plot_cm(test_ds, model)

# **Github Repo:**

https://github.com/KVAlwaysLearning/Age_Emotion_Detection_with_voice_Sub

# **Streamlit App:**

https://ageemotiondetectionwithvoicesub-app.streamlit.app/